# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahidhassanbtk-sys/FlyRank-Interenship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
# Install DuckDB
%pip -q install duckdb huggingface_hub pandas

In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN found successfully")

HF_TOKEN found successfully


In [12]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN found successfully")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB authentication configured.")

HF_TOKEN found successfully
DuckDB authentication configured.


In [16]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset access successful!")
print(info.id)

Dataset access successful!
FlyRank/internship-warehouse


In [17]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("SUCCESS")
    print("Dataset:", info.id)
except Exception as e:
    print("ERROR:")
    print(e)

SUCCESS
Dataset: FlyRank/internship-warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [20]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("SUCCESS: I can access the FlyRank dataset.")
    print("Dataset:", info.id)
except Exception as e:
    print("ACCESS ERROR:")
    print(e)

SUCCESS: I can access the FlyRank dataset.
Dataset: FlyRank/internship-warehouse


## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client on one report date.

**Time window:** March 2026, from `2026-03-01` to `2026-03-31`.

For this assignment, March 2026 is used as the feature/observation window. The final June 2026 sample is not used for development because it is the sealed final month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Query 2 — Row count and date window

I measure the number of rows, clients, and content items in March 2026. I also measure the first and last observed report dates instead of assuming that the month is complete.

In [26]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("SUCCESS: Dataset access is working.")
    print("Dataset:", info.id)
except Exception as e:
    print("DATASET ACCESS ERROR:")
    print(e)

SUCCESS: Dataset access is working.
Dataset: FlyRank/internship-warehouse


In [28]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("✅ SUCCESS: Dataset access is working.")
    print("Dataset:", info.id)
except Exception as e:
    print("❌ DATASET ACCESS ERROR:")
    print(e)

✅ SUCCESS: Dataset access is working.
Dataset: FlyRank/internship-warehouse


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 3 — Data availability

I check GA4 availability using `IS TRUE`. This counts only rows where GA4 availability is explicitly confirmed as true. Rows where the flag is false or NULL are not counted as confirmed available.

In [33]:
import os, getpass
import duckdb

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face read token: ")

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

print("Secret created — ready to query.")

Secret created — ready to query.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This dataset has several limitations.

1. **Unbalanced history:** Different clients can have different amounts of historical data, so observations are not equally supported across all clients.

2. **GA4 availability:** GA4 data is not confirmed for every row. The `ga4_data_available` flag must be checked before treating GA4 measurements as available.

3. **GSC-only early rows:** Some observations can have GSC measurements while GA4 data is not available. These observations should not automatically be treated as having confirmed GA4 data.

4. **Window overlaps:** Fixed 90-day query and performance windows can overlap calendar months, so they should not automatically be interpreted as measurements from only one month.

5. **Future information:** April 2026 information is not available at the March decision moment, so it must not be used as a March feature.

6. **Sealed final month:** The June 2026 `_sample` is the final sealed month and should not be used for development or feature selection.

7. **Decision support:** The data provides observed and measured relationships for decision-support. It does not by itself establish causality.

## 4. Named limitation of this slice

This slice (March 2026, one client's worth of daily performance rows)
has missing values in some GSC/GA4 fields — checked below. A page with
missing `gsc_avg_position` can't have its search position used as a
feature for that day, and rows without GA4 availability can't use
engagement signals. This means some features will only be computable
for a subset of rows, not the full slice — any model trained on GA4-
dependent features implicitly narrows to clients/pages with that access.

In [36]:

from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("SUCCESS: Dataset access is working.")
    print("Dataset:", info.id)
except Exception as e:
    print("DATASET ACCESS ERROR:")
    print(e)

SUCCESS: Dataset access is working.
Dataset: FlyRank/internship-warehouse


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.